## 0. Setup — Local run

Run from the repository root (paths like `Part_2/...` assume it):

```bash
pip install -r Part_15/requirements.txt
jupyter notebook Part_15/Part_15_SmallData_DL_Local.ipynb
```

GPU is used automatically when available (`cuda`); CPU works, training just takes longer.
NOTE: the ChemBERTa section downloads `seyonec/ChemBERTa-zinc-base-v1` from the
HuggingFace hub on first run (needs internet once). If it fails, that section is
skipped gracefully — D-MPNN and GINE still run.

# Part 15: Small-Data Deep Learning for MDM2 Classification

Trains three **small-data-safe** deep-learning models on the 645 MDM2 compounds.
With n=645, big GNNs overfit (Part 12 AttentiveFP ~350k params: AUC 0.924 <
Part 13 Hybrid ~60k params: 0.956 < RF: 0.960), so every model here stays under
~50k trainable params and follows the same protocol as Parts 9/10/12/13:
80/20 stratified split (seed=57), 100 epochs, 10x Y-randomization, 5-fold CV.

**References:**
1. D-MPNN — Yang et al., "Analyzing Learned Molecular Representations for
Property Prediction," *J. Chem. Inf. Model.* 2019 (Chemprop).
Original code: `chemprop/chemprop` (GitHub).
2. GINE — Hu et al., "Strategies for Pre-training Graph Neural Networks,"
*ICLR* 2020. Original code: `snap-stanford/pretrain-gnns` (GitHub).
3. ChemBERTa — Chithrananda et al., "ChemBERTa: Large-Scale Self-Supervised
Pretraining for Molecular Property Prediction," *arXiv:2010.09885* 2020.
Weights: HuggingFace hub `seyonec/ChemBERTa-zinc-base-v1` (frozen here).

**Pipeline:** SMILES → RDKit graphs (78-dim atoms + 6-dim bonds, same as Parts 12/13) →
SMILES-enumeration augmentation (train-only, x4) → D-MPNN-small / GINE-small /
frozen-ChemBERTa + MLP head → 5-fold CV

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import global_add_pool
from torch_geometric.utils import scatter
from torch.utils.data import TensorDataset
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, matthews_corrcoef, roc_curve, auc, confusion_matrix
)
from sklearn.utils import shuffle
from rdkit import Chem
import warnings
warnings.filterwarnings('ignore')

try:
    from transformers import AutoTokenizer, AutoModel
    HAVE_HF = True
except Exception as e:
    HAVE_HF = False
    print(f"transformers not available ({e}) — ChemBERTa section will be skipped.")

RANDOM_SEED = 57
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
print(f"ChemBERTa available: {HAVE_HF}")

## 1. Define Small-Data Architectures

In [ ]:
# === MACCS toggle: set True to fuse 167-bit MACCS keys with the GNN ===
# Morgan fingerprints use the modern rdFingerprintGenerator API (RDKit >= 2023.03).
# For MACCS, MACCSkeys.GenMACCSKeys remains the supported RDKit API
# (rdFingerprintGenerator offers no MACCS generator, verified on RDKit 2026.03).
# NOTE: pretrained .pth files in this repo were trained with USE_MACCS=False.
# Setting USE_MACCS=True changes model shapes -> retrain before loading old weights.
USE_MACCS = False
MACCS_DIM = 167
MACCS_PROJ_DIM = 32


def maccs_from_mol(mol):
    import numpy as np
    from rdkit import DataStructs
    from rdkit.Chem import MACCSkeys
    try:
        bv = MACCSkeys.GenMACCSKeys(mol)
        arr = np.zeros((MACCS_DIM,), dtype=np.float32)
        DataStructs.ConvertToNumpyArray(bv, arr)
        return arr
    except Exception:
        import numpy as np
        return np.zeros((MACCS_DIM,), dtype=np.float32)


def maccs_from_smiles(smi):
    from rdkit import Chem
    import numpy as np
    try:
        mol = Chem.MolFromSmiles(str(smi))
        if mol is None:
            return np.zeros((MACCS_DIM,), dtype=np.float32)
        return maccs_from_mol(mol)
    except Exception:
        return np.zeros((MACCS_DIM,), dtype=np.float32)


In [ ]:
class DirectedMPNN(nn.Module):
    """Directed message-passing network after Yang et al. 2019 (Chemprop).

    Messages live on directed bonds; each update excludes the reverse edge
    (no tottering), with a skip connection from the initial edge state.
    hidden=64, depth=2 -> ~23k params (small-data-safe)."""

    def __init__(self, num_node_features=78, num_edge_features=6, hidden_dim=64,
                 depth=2, dropout=0.2, num_classes=2,
                 use_maccs=False, maccs_dim=167, maccs_proj_dim=32):
        super().__init__()
        self.depth = depth
        self.dropout = dropout
        self.edge_init = nn.Linear(num_node_features + num_edge_features, hidden_dim)
        self.edge_update = nn.Linear(hidden_dim, hidden_dim)
        self.atom_update = nn.Linear(num_node_features + hidden_dim, hidden_dim)
        # MACCS fusion (off by default -> identical to the original model)
        self.use_maccs = use_maccs
        if use_maccs:
            self.maccs_proj = nn.Sequential(
                nn.Linear(maccs_dim, maccs_proj_dim), nn.ReLU())
            self.ffn = nn.Sequential(
                nn.Linear(hidden_dim + maccs_proj_dim, hidden_dim),
                nn.ReLU(),
                nn.Dropout(dropout),
                nn.Linear(hidden_dim, num_classes)
            )
        else:
            self.ffn = nn.Sequential(
                nn.Linear(hidden_dim, hidden_dim),
                nn.ReLU(),
                nn.Dropout(dropout),
                nn.Linear(hidden_dim, num_classes)
            )

    def forward(self, x, edge_index, edge_attr, batch, maccs=None):
        src, dst = edge_index
        n_nodes = x.size(0)
        n_edges = src.size(0)
        # reverse-edge map: rev[i] = index of edge (dst[i] -> src[i])
        key = src * n_nodes + dst
        rkey = dst * n_nodes + src
        order = torch.argsort(key)
        pos = torch.searchsorted(key[order], rkey)
        rev = order[pos.clamp(max=n_edges - 1)]

        h0 = F.relu(self.edge_init(torch.cat([x[src], edge_attr], dim=1)))
        h = h0
        for _ in range(self.depth):
            incoming = scatter(h, dst, dim=0, dim_size=n_nodes, reduce='sum')
            m = incoming[src] - h[rev]
            h = F.relu(h0 + F.dropout(self.edge_update(m), p=self.dropout,
                                      training=self.training))
        atom_in = scatter(h, dst, dim=0, dim_size=n_nodes, reduce='sum')
        a = F.relu(self.atom_update(torch.cat([x, atom_in], dim=1)))
        g = global_add_pool(a, batch)
        if self.use_maccs:
            if maccs is not None:
                mm = maccs.to(dtype=g.dtype, device=g.device)
                g = torch.cat([g, self.maccs_proj(mm)], dim=1)
            else:
                g = torch.cat([g, torch.zeros((g.size(0), 32), device=g.device, dtype=g.dtype)], dim=1)
        return self.ffn(g)


In [ ]:
class GINELayer(nn.Module):
    """Single GIN-with-edge-features layer after Hu et al. 2020.

    h_v = MLP_node((1 + eps) * h_v + sum_{u in N(v)} (h_u + MLP_edge(e_uv)))."""

    def __init__(self, in_dim, hidden_dim=64, edge_dim=6):
        super().__init__()
        self.proj = nn.Linear(in_dim, hidden_dim) if in_dim != hidden_dim else nn.Identity()
        self.edge_mlp = nn.Sequential(nn.Linear(edge_dim, hidden_dim), nn.ReLU())
        self.node_mlp = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim)
        )
        self.eps = nn.Parameter(torch.zeros(1))

    def forward(self, x, edge_index, edge_attr):
        src, dst = edge_index
        h = self.proj(x)
        msg = scatter(h[src] + self.edge_mlp(edge_attr), dst,
                      dim=0, dim_size=x.size(0), reduce='sum')
        return self.node_mlp((1.0 + self.eps) * h + msg)


class GINESmall(nn.Module):
    """3-layer GINE, hidden=64 -> ~40k params (small-data-safe)."""

    def __init__(self, num_node_features=78, num_edge_features=6, hidden_dim=64,
                 num_layers=3, dropout=0.3, num_classes=2,
                 use_maccs=False, maccs_dim=167, maccs_proj_dim=32):
        super().__init__()
        self.layers = nn.ModuleList([
            GINELayer(num_node_features if i == 0 else hidden_dim, hidden_dim, num_edge_features)
            for i in range(num_layers)
        ])
        self.dropout = dropout
        # MACCS fusion (off by default -> identical to the original model)
        self.use_maccs = use_maccs
        if use_maccs:
            self.maccs_proj = nn.Sequential(
                nn.Linear(maccs_dim, maccs_proj_dim), nn.ReLU())
            self.head = nn.Sequential(
                nn.Linear(hidden_dim + maccs_proj_dim, 32),
                nn.ReLU(),
                nn.Dropout(dropout),
                nn.Linear(32, num_classes)
            )
        else:
            self.head = nn.Sequential(
                nn.Linear(hidden_dim, 32),
                nn.ReLU(),
                nn.Dropout(dropout),
                nn.Linear(32, num_classes)
            )

    def forward(self, x, edge_index, edge_attr, batch, maccs=None):
        for layer in self.layers:
            x = F.relu(layer(x, edge_index, edge_attr))
            x = F.dropout(x, p=self.dropout, training=self.training)
        g = global_add_pool(x, batch)
        if self.use_maccs:
            if maccs is not None:
                m = maccs.to(dtype=g.dtype, device=g.device)
                g = torch.cat([g, self.maccs_proj(m)], dim=1)
            else:
                g = torch.cat([g, torch.zeros((g.size(0), 32), device=g.device, dtype=g.dtype)], dim=1)
        return self.head(g)


In [ ]:
class ChemBERTaMLP(nn.Module):
    """Trainable head on FROZEN ChemBERTa embeddings (Chithrananda et al. 2020).

    The language model (pretrained on ~77M SMILES from ZINC) is never updated;
    only this 768->64->2 head trains (~49k params), so 645 compounds suffice."""

    def __init__(self, in_dim=768, hidden_dim=64, dropout=0.3, num_classes=2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes)
        )

    def forward(self, emb):
        return self.net(emb)


def chemberta_embed(smiles_list, tok, lm, batch_size=64):
    """Mean-pooled last-hidden-state embeddings (frozen, no grad)."""
    lm.eval()
    embs = []
    with torch.no_grad():
        for i in range(0, len(smiles_list), batch_size):
            enc = tok(smiles_list[i:i + batch_size], padding=True, truncation=True,
                      max_length=512, return_tensors='pt')
            enc = {k: v.to(device) for k, v in enc.items()}
            h = lm(**enc).last_hidden_state
            mask = enc['attention_mask'].unsqueeze(-1)
            embs.append(((h * mask).sum(1) / mask.sum(1).clamp(min=1)).cpu())
    return torch.cat(embs)

## 2. Load Data + Featurize

In [ ]:
df = pd.read_csv('Part_2/ro5_properties_filtered.csv')
df['y'] = df['bioactivity_class'].apply(lambda x: 1 if x == 'Active' else 0)
print(f"Dataset: {len(df)} compounds")
print(f"Class distribution:\n{df['y'].value_counts()}")

In [ ]:
ATOM_CHOICES = {
    'atomic_num': list(range(1, 101)),
    'degree': [0, 1, 2, 3, 4, 5],
    'formal_charge': [-2, -1, 0, 1, 2, 3],
    'num_hs': [0, 1, 2, 3, 4],
    'hybridization': [
        Chem.rdchem.HybridizationType.SP, Chem.rdchem.HybridizationType.SP2,
        Chem.rdchem.HybridizationType.SP3, Chem.rdchem.HybridizationType.SP3D,
        Chem.rdchem.HybridizationType.SP3D2
    ]
}

def one_hot(val, choices):
    enc = [0] * len(choices)
    if val in choices:
        enc[choices.index(val)] = 1
    return enc

def atom_features(atom):
    f = []
    f += one_hot(atom.GetAtomicNum(), ATOM_CHOICES['atomic_num'])
    f += one_hot(atom.GetTotalDegree(), ATOM_CHOICES['degree'])
    f += one_hot(atom.GetFormalCharge(), ATOM_CHOICES['formal_charge'])
    f += one_hot(atom.GetTotalNumHs(), ATOM_CHOICES['num_hs'])
    f += one_hot(atom.GetHybridization(), ATOM_CHOICES['hybridization'])
    f.append(int(atom.GetIsAromatic()))
    f.append(int(atom.IsInRing()))
    return (f + [0] * 78)[:78]

def bond_features(bond):
    bt = bond.GetBondType()
    return [int(bt == Chem.rdchem.BondType.SINGLE),
            int(bt == Chem.rdchem.BondType.DOUBLE),
            int(bt == Chem.rdchem.BondType.TRIPLE),
            int(bt == Chem.rdchem.BondType.AROMATIC),
            int(bond.GetIsConjugated()),
            int(bond.IsInRing())]

def mol_to_graph(smiles, label):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    node_features = [atom_features(a) for a in mol.GetAtoms()]
    edge_index, edge_attr = [], []
    for b in mol.GetBonds():
        i, j = b.GetBeginAtomIdx(), b.GetEndAtomIdx()
        bf = bond_features(b)
        edge_index.extend([[i, j], [j, i]])
        edge_attr.extend([bf, bf])
    if not edge_index:
        edge_index = [[0, 0]]
        edge_attr = [[0] * 6]
    try:
        maccs_arr = maccs_from_mol(mol)
    except Exception:
        import numpy as _np
        maccs_arr = _np.zeros((MACCS_DIM,), dtype=_np.float32)
    return Data(
        maccs=torch.tensor(maccs_arr, dtype=torch.float).unsqueeze(0),
        x=torch.tensor(node_features, dtype=torch.float),
        edge_index=torch.tensor(edge_index, dtype=torch.long).t().contiguous(),
        edge_attr=torch.tensor(edge_attr, dtype=torch.float),
        y=torch.tensor([label], dtype=torch.long)
    )


In [ ]:
graphs = []
for _, row in df.iterrows():
    g = mol_to_graph(row['canonical_smiles'], row['y'])
    if g is not None:
        graphs.append(g)

labels = np.array([g.y.item() for g in graphs])
print(f"Converted {len(graphs)}/{len(df)} molecules to graphs")
print(f"Example: {graphs[0].x.shape[0]} atoms, {graphs[0].edge_index.shape[1]} edges, edge_attr {tuple(graphs[0].edge_attr.shape)}")

## 3. Train/Test Split

In [ ]:
train_idx, test_idx, y_train, y_test = train_test_split(
    np.arange(len(graphs)), labels, test_size=0.2, random_state=RANDOM_SEED, stratify=labels
)
train_graphs = [graphs[i] for i in train_idx]
test_graphs = [graphs[i] for i in test_idx]
train_loader = DataLoader(train_graphs, batch_size=64, shuffle=True)
test_loader = DataLoader(test_graphs, batch_size=64, shuffle=False)
print(f"Train: {len(train_graphs)}, Test: {len(test_graphs)}")

## 4. SMILES Augmentation (train-only, x4)

Same molecule, several valid SMILES spellings (`MolToSmiles(doRandom=True)`).
Only the TRAIN pool is augmented — test/CV/Y-rand stay canonical, so the
honesty checks remain comparable with Parts 9/10/12/13.

In [ ]:
def enumerated_smiles(smi, n=4):
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        return [smi]
    outs = []
    for seed in range(n * 5):
        try:
            s = Chem.MolToSmiles(mol, doRandom=True, canonical=False,
                                 rootedAtAtom=seed % max(1, mol.GetNumAtoms()))
            if s not in outs:
                outs.append(s)
            if len(outs) >= n:
                break
        except Exception:
            pass
    if not outs:
        outs = [Chem.MolToSmiles(mol)]
    return outs

AUGMENT = 4
train_smiles = df.iloc[train_idx]['canonical_smiles'].tolist()
aug_pairs = [(s, int(y)) for smi, y in zip(train_smiles, y_train)
             for s in enumerated_smiles(smi, AUGMENT)]

aug_graphs = []
for s, y in aug_pairs:
    g = mol_to_graph(s, y)
    if g is not None:
        aug_graphs.append(g)
aug_loader = DataLoader(aug_graphs, batch_size=64, shuffle=True)
print(f"Augmented train pool: {len(aug_graphs)} graphs ({len(aug_pairs)} enumerated SMILES)")

## 5. Training Utilities

In [ ]:
def train_epoch_graph(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0
    for batch in loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        out = model(batch.x, batch.edge_index, batch.edge_attr, batch.batch, getattr(batch, 'maccs', None) if USE_MACCS else None)
        loss = criterion(out, batch.y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * batch.num_graphs
    return total_loss / len(loader.dataset)

def evaluate_graph(model, loader):
    model.eval()
    preds, probs, labels = [], [], []
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            out = model(batch.x, batch.edge_index, batch.edge_attr, batch.batch, getattr(batch, 'maccs', None) if USE_MACCS else None)
            probs.extend(F.softmax(out, dim=1)[:, 1].cpu().numpy())
            preds.extend(out.argmax(dim=1).cpu().numpy())
            labels.extend(batch.y.cpu().numpy())
    return np.array(preds), np.array(probs), np.array(labels)

def train_epoch_mlp(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * yb.size(0)
    return total_loss / len(loader.dataset)

def evaluate_mlp(model, loader):
    model.eval()
    preds, probs, labels = [], [], []
    with torch.no_grad():
        for xb, yb in loader:
            out = model(xb.to(device))
            probs.extend(F.softmax(out, dim=1)[:, 1].cpu().numpy())
            preds.extend(out.argmax(dim=1).cpu().numpy())
            labels.extend(yb.numpy())
    return np.array(preds), np.array(probs), np.array(labels)

def metric_row(name, pred, prob, true):
    b = (prob > 0.5).astype(int)
    return {
        'Model': name,
        'Accuracy': accuracy_score(true, b),
        'Precision': precision_score(true, b, zero_division=0),
        'F1-score': f1_score(true, b, zero_division=0),
        'Sensitivity': recall_score(true, b, zero_division=0),
        'ROC-AUC': roc_auc_score(true, prob),
        'MCC': matthews_corrcoef(true, b)
    }


## 6. Train D-MPNN-small

In [ ]:
dmpnn = DirectedMPNN(num_node_features=78, num_edge_features=6, hidden_dim=64,
                     depth=2, dropout=0.2, use_maccs=USE_MACCS).to(device)
dmpnn_opt = torch.optim.Adam(dmpnn.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()
print(f"D-MPNN parameters: {sum(p.numel() for p in dmpnn.parameters()):,}")

EPOCHS = 100
dmpnn_losses, dmpnn_aucs = [], []

for epoch in range(1, EPOCHS + 1):
    loss = train_epoch_graph(dmpnn, aug_loader, dmpnn_opt, criterion)
    dmpnn_losses.append(loss)
    if epoch % 10 == 0 or epoch == 1:
        _, probs, true = evaluate_graph(dmpnn, test_loader)
        auc_val = roc_auc_score(true, probs) if len(np.unique(true)) > 1 else 0
        dmpnn_aucs.append(auc_val)
        print(f"Epoch {epoch:3d}/{EPOCHS} | Loss: {loss:.4f} | Test AUC: {auc_val:.3f}")

print("\nD-MPNN training complete.")


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(dmpnn_losses, 'b-', lw=1.5); ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax1.set_title('Training Loss (D-MPNN)'); ax1.grid(True, alpha=0.3)
ax2.plot(dmpnn_aucs, 'r-o', lw=1.5)
ax2.set_xlabel('Epoch (x10)'); ax2.set_ylabel('ROC-AUC'); ax2.set_title('Test ROC-AUC (D-MPNN)'); ax2.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('training_curves_dmpnn.png', dpi=300, bbox_inches='tight')
plt.show()

## 7. Train GINE-small

In [ ]:
gine = GINESmall(num_node_features=78, num_edge_features=6, hidden_dim=64,
                 num_layers=3, dropout=0.3, use_maccs=USE_MACCS).to(device)
gine_opt = torch.optim.Adam(gine.parameters(), lr=1e-3)
print(f"GINE parameters: {sum(p.numel() for p in gine.parameters()):,}")

gine_losses, gine_aucs = [], []

for epoch in range(1, EPOCHS + 1):
    loss = train_epoch_graph(gine, aug_loader, gine_opt, criterion)
    gine_losses.append(loss)
    if epoch % 10 == 0 or epoch == 1:
        _, probs, true = evaluate_graph(gine, test_loader)
        auc_val = roc_auc_score(true, probs) if len(np.unique(true)) > 1 else 0
        gine_aucs.append(auc_val)
        print(f"Epoch {epoch:3d}/{EPOCHS} | Loss: {loss:.4f} | Test AUC: {auc_val:.3f}")

print("\nGINE training complete.")


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(gine_losses, 'b-', lw=1.5); ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax1.set_title('Training Loss (GINE)'); ax1.grid(True, alpha=0.3)
ax2.plot(gine_aucs, 'r-o', lw=1.5)
ax2.set_xlabel('Epoch (x10)'); ax2.set_ylabel('ROC-AUC'); ax2.set_title('Test ROC-AUC (GINE)'); ax2.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('training_curves_gine.png', dpi=300, bbox_inches='tight')
plt.show()

## 8. Frozen ChemBERTa Embeddings + MLP Head

In [ ]:
HAVE_CB = False
cb_tok, cb_lm = None, None
if HAVE_HF:
    try:
        cb_tok = AutoTokenizer.from_pretrained('seyonec/ChemBERTa-zinc-base-v1')
        cb_lm = AutoModel.from_pretrained('seyonec/ChemBERTa-zinc-base-v1').to(device)
        for p in cb_lm.parameters():
            p.requires_grad = False
        HAVE_CB = True
        print("ChemBERTa loaded and frozen.")
    except Exception as e:
        print(f"ChemBERTa download failed ({e}) — skipping ChemBERTa section.")
else:
    print("transformers missing — skipping ChemBERTa section.")

In [ ]:
if HAVE_CB:
    all_smiles = df['canonical_smiles'].tolist()
    aug_smiles = [s for s, _ in aug_pairs]
    emb_all = chemberta_embed(all_smiles, cb_tok, cb_lm)
    emb_aug = chemberta_embed(aug_smiles, cb_tok, cb_lm)
    print(f"Embeddings: all {tuple(emb_all.shape)}, aug {tuple(emb_aug.shape)}")
    torch.save({'all': emb_all, 'aug': emb_aug}, 'chemberta_embeddings.pt')
    print("Embeddings cached: Part_15/chemberta_embeddings.pt")
    # MACCS flag: optionally fuse 167-bit MACCS keys with the frozen embeddings.
    if USE_MACCS:
        maccs_all = torch.tensor(np.stack([maccs_from_smiles(s) for s in all_smiles]), dtype=torch.float)
        maccs_aug = torch.tensor(np.stack([maccs_from_smiles(s) for s in aug_smiles]), dtype=torch.float)
        emb_all = torch.cat([emb_all, maccs_all], dim=1)
        emb_aug = torch.cat([emb_aug, maccs_aug], dim=1)
        print(f"USE_MACCS=True -> ChemBERTa inputs widened to {tuple(emb_all.shape)}")

    emb_train = emb_all[train_idx]
    emb_test = emb_all[test_idx]
    y_tr = torch.tensor(np.array(y_train), dtype=torch.long)
    y_te = torch.tensor(np.array(y_test), dtype=torch.long)
    aug_y = torch.tensor([y for _, y in aug_pairs], dtype=torch.long)
    print(f"emb_aug {len(emb_aug)} vs aug labels {len(aug_y)}")
    mlp_train_loader = DataLoader(TensorDataset(emb_aug, aug_y), batch_size=64, shuffle=True)
    mlp_test_loader = DataLoader(TensorDataset(emb_test, y_te), batch_size=64, shuffle=False)


In [ ]:
if HAVE_CB:
    cb_mlp = ChemBERTaMLP(in_dim=emb_all.shape[1], hidden_dim=64, dropout=0.3).to(device)
    cb_opt = torch.optim.Adam(cb_mlp.parameters(), lr=1e-3)
    print(f"ChemBERTa-MLP trainable parameters: {sum(p.numel() for p in cb_mlp.parameters()):,}")

    cb_losses, cb_aucs = [], []
    for epoch in range(1, EPOCHS + 1):
        loss = train_epoch_mlp(cb_mlp, mlp_train_loader, cb_opt, criterion)
        cb_losses.append(loss)
        if epoch % 10 == 0 or epoch == 1:
            _, probs, true = evaluate_mlp(cb_mlp, mlp_test_loader)
            auc_val = roc_auc_score(true, probs) if len(np.unique(true)) > 1 else 0
            cb_aucs.append(auc_val)
            print(f"Epoch {epoch:3d}/{EPOCHS} | Loss: {loss:.4f} | Test AUC: {auc_val:.3f}")

    print("\nChemBERTa-MLP training complete.")

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    ax1.plot(cb_losses, 'b-', lw=1.5); ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
    ax1.set_title('Training Loss (ChemBERTa-MLP)'); ax1.grid(True, alpha=0.3)
    ax2.plot(cb_aucs, 'r-o', lw=1.5)
    ax2.set_xlabel('Epoch (x10)'); ax2.set_ylabel('ROC-AUC'); ax2.set_title('Test ROC-AUC (ChemBERTa-MLP)'); ax2.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('training_curves_chemberta.png', dpi=300, bbox_inches='tight')
    plt.show()

## 9. Test Set Evaluation (all three)

In [ ]:
part15_rows = []

p, pr, t = evaluate_graph(dmpnn, test_loader)
part15_rows.append(metric_row('D-MPNN', p, pr, t))
dmpnn_prob, dmpnn_true = pr, t

p, pr, t = evaluate_graph(gine, test_loader)
part15_rows.append(metric_row('GINE', p, pr, t))
gine_prob, gine_true = pr, t

if HAVE_CB:
    p, pr, t = evaluate_mlp(cb_mlp, mlp_test_loader)
    part15_rows.append(metric_row('ChemBERTa-MLP', p, pr, t))
    cb_prob, cb_true = pr, t

test_results15 = pd.DataFrame(part15_rows).round(3)
test_results15.to_csv('performance_part15_test.csv', index=False)
test_results15

## 10. ROC Curves

In [ ]:
plt.figure(figsize=(8, 6))
for name, prob, true in ([('D-MPNN', dmpnn_prob, dmpnn_true),
                          ('GINE', gine_prob, gine_true)] +
                         ([('ChemBERTa-MLP', cb_prob, cb_true)] if HAVE_CB else [])):
    fpr, tpr, _ = roc_curve(true, prob)
    plt.plot(fpr, tpr, lw=2, label=f'{name} (AUC = {auc(fpr, tpr):.3f})')
plt.plot([0, 1], [0, 1], 'k--', lw=2, label='Random')
plt.xlim([0, 1]); plt.ylim([0, 1.05])
plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
plt.title('ROC Curves - Part 15 models')
plt.legend(loc='lower right')
plt.tight_layout()
plt.savefig('roc_curve_part15.png', dpi=300, bbox_inches='tight')
plt.show()

## 11. Confusion Matrices

In [ ]:
cm_models = [('D-MPNN', dmpnn_prob, dmpnn_true), ('GINE', gine_prob, gine_true)]
if HAVE_CB:
    cm_models.append(('ChemBERTa-MLP', cb_prob, cb_true))

fig, axes = plt.subplots(1, len(cm_models), figsize=(6 * len(cm_models), 5))
if len(cm_models) == 1:
    axes = [axes]
for ax, (name, prob, true) in zip(axes, cm_models):
    cm = confusion_matrix(true, (prob > 0.5).astype(int))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Inactive', 'Active'], yticklabels=['Inactive', 'Active'], ax=ax)
    ax.set_xlabel('Predicted'); ax.set_ylabel('True'); ax.set_title(f'Confusion Matrix ({name})')
plt.tight_layout()
plt.savefig('confusion_matrix_part15.png', dpi=300, bbox_inches='tight')
plt.show()

## 12. Y-Randomization Test

In [ ]:
def fresh_dmpnn():
    return DirectedMPNN(num_node_features=78, num_edge_features=6, hidden_dim=64,
                        depth=2, dropout=0.2, use_maccs=USE_MACCS).to(device)

def fresh_gine():
    return GINESmall(num_node_features=78, num_edge_features=6, hidden_dim=64,
                     num_layers=3, dropout=0.3, use_maccs=USE_MACCS).to(device)

def fresh_mlp(in_dim=768):
    return ChemBERTaMLP(in_dim=in_dim, hidden_dim=64, dropout=0.3).to(device)

shuffled_results15 = []
for i in range(10):
    y_sh = shuffle(y_train, random_state=i)
    sg = [Data(x=g.x, edge_index=g.edge_index, edge_attr=g.edge_attr, **({'maccs': g.maccs} if hasattr(g, 'maccs') else {}), y=torch.tensor([y_sh[j]], dtype=torch.long))
          for j, g in enumerate(train_graphs)]
    sl = DataLoader(sg, batch_size=64, shuffle=True)

    for name, fresh in [('D-MPNN', fresh_dmpnn), ('GINE', fresh_gine)]:
        m = fresh()
        o = torch.optim.Adam(m.parameters(), lr=1e-3)
        for _ in range(100):
            train_epoch_graph(m, sl, o, criterion)
        p, pr, t = evaluate_graph(m, test_loader)
        b = (pr > 0.5).astype(int)
        shuffled_results15.append({
            'Model': f'Shuffled {name} {i+1}',
            'Accuracy': round(accuracy_score(t, b), 3),
            'F1-score': round(f1_score(t, b, zero_division=0), 3),
            'ROC-AUC': round(roc_auc_score(t, pr), 3),
            'MCC': round(matthews_corrcoef(t, b), 3)
        })
    if HAVE_CB:
        m = fresh_mlp(emb_all.shape[1])
        o = torch.optim.Adam(m.parameters(), lr=1e-3)
        sh_loader = DataLoader(TensorDataset(emb_train, torch.tensor(np.array(y_sh), dtype=torch.long)),
                               batch_size=64, shuffle=True)
        for _ in range(100):
            train_epoch_mlp(m, sh_loader, o, criterion)
        p, pr, t = evaluate_mlp(m, mlp_test_loader)
        b = (pr > 0.5).astype(int)
        shuffled_results15.append({
            'Model': f'Shuffled ChemBERTa-MLP {i+1}',
            'Accuracy': round(accuracy_score(t, b), 3),
            'F1-score': round(f1_score(t, b, zero_division=0), 3),
            'ROC-AUC': round(roc_auc_score(t, pr), 3),
            'MCC': round(matthews_corrcoef(t, b), 3)
        })
    print(f"Shuffle {i+1}/10 done")

shuffled_df15 = pd.DataFrame(shuffled_results15)
shuffled_df15.to_csv('performance_part15_shuffled.csv', index=False)
shuffled_df15


## 13. 5-Fold Cross-Validation

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
cv_rows15 = []
all_labels = np.array([g.y.item() for g in graphs])

for fold, (tr_i, va_i) in enumerate(skf.split(graphs, all_labels)):
    print(f"Fold {fold + 1}/5")
    tr_loader = DataLoader([graphs[i] for i in tr_i], batch_size=64, shuffle=True)
    va_loader = DataLoader([graphs[i] for i in va_i], batch_size=64, shuffle=False)

    for name, fresh in [('D-MPNN', fresh_dmpnn), ('GINE', fresh_gine)]:
        m = fresh()
        o = torch.optim.Adam(m.parameters(), lr=1e-3)
        for _ in range(100):
            train_epoch_graph(m, tr_loader, o, criterion)
        p, pr, t = evaluate_graph(m, va_loader)
        b = (pr > 0.5).astype(int)
        cv_rows15.append({'Model': name, 'Fold': fold + 1,
                          'Accuracy': accuracy_score(t, b),
                          'F1-score': f1_score(t, b, zero_division=0),
                          'ROC-AUC': roc_auc_score(t, pr),
                          'MCC': matthews_corrcoef(t, b)})
        print(f"  {name} AUC: {cv_rows15[-1]['ROC-AUC']:.3f} | F1: {cv_rows15[-1]['F1-score']:.3f}")

    if HAVE_CB:
        m = fresh_mlp(emb_all.shape[1])
        o = torch.optim.Adam(m.parameters(), lr=1e-3)
        f_tr = DataLoader(TensorDataset(emb_all[tr_i],
                                        torch.tensor(all_labels[tr_i], dtype=torch.long)),
                          batch_size=64, shuffle=True)
        f_va = DataLoader(TensorDataset(emb_all[va_i],
                                        torch.tensor(all_labels[va_i], dtype=torch.long)),
                          batch_size=64, shuffle=False)
        for _ in range(100):
            train_epoch_mlp(m, f_tr, o, criterion)
        p, pr, t = evaluate_mlp(m, f_va)
        b = (pr > 0.5).astype(int)
        cv_rows15.append({'Model': 'ChemBERTa-MLP', 'Fold': fold + 1,
                          'Accuracy': accuracy_score(t, b),
                          'F1-score': f1_score(t, b, zero_division=0),
                          'ROC-AUC': roc_auc_score(t, pr),
                          'MCC': matthews_corrcoef(t, b)})
        print(f"  ChemBERTa-MLP AUC: {cv_rows15[-1]['ROC-AUC']:.3f} | F1: {cv_rows15[-1]['F1-score']:.3f}")

cv_results15 = pd.DataFrame(cv_rows15)
cv_results15.to_csv('performance_part15_cv.csv', index=False)
print(cv_results15.groupby('Model')[['Accuracy', 'F1-score', 'ROC-AUC', 'MCC']].agg(['mean', 'std']).round(3))
cv_results15

## 14. Save Models + Compare

In [ ]:
torch.save(dmpnn.state_dict(), 'dmpnn_mdm2.pth')
print("Model saved: Part_15/dmpnn_mdm2.pth")
torch.save(gine.state_dict(), 'gine_mdm2.pth')
print("Model saved: Part_15/gine_mdm2.pth")
if HAVE_CB:
    torch.save(cb_mlp.state_dict(), 'chemberta_mlp_mdm2.pth')
    print("Model saved: Part_15/chemberta_mlp_mdm2.pth")

In [ ]:
rows = []

def try_read(cands):
    for f in cands:
        try:
            return pd.read_csv(f)
        except Exception:
            pass
    return None

rf = try_read(['Part_6/performance_morgan_tuned.csv', 'performance_morgan_tuned.csv'])
if rf is not None:
    r = rf[rf['Model'] == 'RandomForestClassifier'].iloc[0].to_dict()
    if 'Sensitivity (Recall)' in r:
        r['Sensitivity'] = r.pop('Sensitivity (Recall)')
    r['Model'] = 'RF (Morgan) - Part 6'; rows.append(r)
for fnames, label in [(['Part_9/performance_gcn_test.csv', 'performance_gcn_test.csv'], 'GCN (scratch) - Part 9'),
                      (['Part_10/performance_pretrained_gnn_test.csv', 'performance_pretrained_gnn_test.csv'], 'GIN - Part 10'),
                      (['Part_12/performance_attentivefp_test.csv', 'performance_attentivefp_test.csv'], 'AttentiveFP - Part 12'),
                      (['Part_13/performance_hybrid_test.csv', 'performance_hybrid_test.csv'], 'HybridGCNGAT - Part 13')]:
    t = try_read(fnames)
    if t is not None:
        r = t.iloc[0].to_dict(); r['Model'] = label; rows.append(r)
for _, r0 in test_results15.iterrows():
    r = r0.to_dict(); r['Model'] = r['Model'] + ' - Part 15'; rows.append(r)

comparison15 = pd.DataFrame(rows).round(3)
comparison15.to_csv('comparison_part15_vs_all.csv', index=False)

plot_df = comparison15.dropna(subset=['ROC-AUC']).sort_values('ROC-AUC')
fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(plot_df['Model'], plot_df['ROC-AUC'], color='mediumseagreen')
ax.set_xlabel('ROC-AUC'); ax.set_title('All models by test ROC-AUC')
for i, v in enumerate(plot_df['ROC-AUC']):
    ax.text(v + 0.005, i, f'{v:.3f}', va='center', fontsize=9)
plt.tight_layout()
plt.savefig('comparison_part15_vs_all.png', dpi=300, bbox_inches='tight')
plt.show()
comparison15

In [ ]:
print("\n=== Part 15 Complete ===\n")
print("Outputs saved to Part_15/")